In [1]:
import numpy as np
import pandas as pd

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [4]:
df = pd.read_csv('covid_toy.csv')
df

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No
...,...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore,No
96,51,Female,101.0,Strong,Kolkata,Yes
97,20,Female,101.0,Mild,Bangalore,No
98,5,Female,98.0,Strong,Mumbai,No


In [5]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [6]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [9]:
X_train

,age,gender,fever,cough,city
2,42,Male,101.0,Mild,Delhi
46,19,Female,101.0,Mild,Mumbai
41,82,Male,NaN,Mild,Kolkata
36,38,Female,101.0,Mild,Bangalore
64,42,Male,104.0,Mild,Mumbai
...,...,...,...,...,...
21,73,Male,98.0,Mild,Bangalore
47,18,Female,104.0,Mild,Bangalore
57,49,Female,99.0,Strong,Bangalore
20,12,Male,98.0,Strong,Bangalore


In [11]:
# Normal Way(long Way)

# adding simple imputer to fever col. to fill the missing values.
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])

In [12]:
X_train_fever.shape

(80, 1)

In [13]:
X_test_fever.shape

(20, 1)

In [15]:
# Ordinalencoder  -> cough
oe = OrdinalEncoder(categories = [['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

X_test_cough = oe.fit_transform(X_test[['cough']])

In [16]:
X_train_cough.shape

(80, 1)

In [19]:
# OneHotEncoder -> gender, city

ohe = OneHotEncoder(drop="first",sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

In [22]:
X_train_gender_city.shape

(80, 4)

In [24]:
# Extracting age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

In [25]:
X_train_age.shape

(80, 1)

In [26]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

In [28]:
X_train_transformed.shape

(80, 7)

In [40]:
X_test_transformed.shape

(20, 6)

In [29]:
# we have to do all the steps.
# if we want to transform the data , in which we have to do diff. transformation

In [30]:
# but if we know ColumnTransform , we don't need to do this all, it will do it by itself.

In [33]:
from sklearn.compose import ColumnTransformer

In [35]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
], remainder='passthrough')

In [36]:
transformer.fit_transform(X_train)

array([[101.        ,   0.        ,   1.        ,   1.        ,
          0.        ,   0.        ,  42.        ],
       [101.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  19.        ],
       [100.83333333,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  82.        ],
       [101.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  38.        ],
       [104.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   1.        ,  42.        ],
       [101.        ,   0.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  83.        ],
       [104.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  34.        ],
       [103.        ,   0.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  69.        ],
       [101.        ,   1.        ,   1.        ,   0.        ,
          0.    

In [37]:
transformer.fit_transform(X_train).shape

(80, 7)

In [38]:
transformer.fit_transform(X_test)

array([[101.        ,   1.        ,   1.        ,   0.        ,
          0.        ,  47.        ],
       [ 98.        ,   0.        ,   1.        ,   1.        ,
          0.        ,  83.        ],
       [100.        ,   1.        ,   0.        ,   0.        ,
          0.        ,  19.        ],
       [ 98.        ,   1.        ,   1.        ,   0.        ,
          1.        ,  34.        ],
       [101.        ,   0.        ,   0.        ,   0.        ,
          0.        ,  20.        ],
       [ 99.        ,   1.        ,   0.        ,   0.        ,
          1.        ,  25.        ],
       [100.        ,   0.        ,   1.        ,   1.        ,
          0.        ,  27.        ],
       [103.        ,   0.        ,   0.        ,   0.        ,
          0.        ,  16.        ],
       [100.        ,   0.        ,   1.        ,   0.        ,
          1.        ,  55.        ],
       [100.        ,   1.        ,   0.        ,   0.        ,
          1.        ,  11. 

In [39]:
transformer.fit_transform(X_test).shape

(20, 6)